# Week 4 & 5 - Data Cleaning and Preparation

Prepare the enriched MLS datasets for analysis by fixing data types, handling invalid values, validating date consistency, and checking geographic data quality.

In [1]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns


### Task 1: Convert date fields to datetime format
(CloseDate, PurchaseContractDate, ListingContractDate,
ContractStatusChangeDate)

In [2]:
data_dir = Path("../data/cleaned_csv")

sold = pd.read_csv(data_dir / "sold_residential_with_rates.csv", low_memory=False)
listing = pd.read_csv(data_dir / "listings_residential_with_rates.csv",low_memory=False)

In [3]:
print("Sold shape:", sold.shape)
print("Listing shape:", listing.shape)

Sold shape: (414054, 78)
Listing shape: (547162, 78)


In [4]:
date_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]


for column in date_fields:
    if column in listing.columns: # for LISTING
        listing[column] = pd.to_datetime(listing[column], errors="coerce")


for column in date_fields:
    if column in sold.columns: # for SOLD
        sold[column] = pd.to_datetime(sold[column], errors="coerce")
        sold[column] = pd.to_datetime(sold[column], errors="coerce")

In [5]:
sold[[column for column in date_fields if column in sold.columns]].dtypes

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object

In [6]:
listing[[column for column in date_fields if column in listing.columns]].dtypes

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object

In [7]:
listing.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'PropertyType.1', 'MlsStatus',
       'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BuildingAreaTotal', 'BedroomsTot

In [8]:
# there are duplicate columns that i have from merging or something from beforehand, just drop these
listing_drop_columns = [
    "PropertyType.1",
    "ListAgentFirstName.1",
    "DaysOnMarket.1",
    "LivingArea.1",
    "Longitude.1",
    "Latitude.1",
    "ListPrice.1",
    "ListAgentLastName.1",
    "CloseDate.1",
    "UnparsedAddress.1"
]

listing = listing.drop(
    columns=[
        column
        for column in listing_drop_columns
        if column in listing.columns
    ]
)

In [9]:
listing.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate',
       'CoBuyerAgentFirstName', 'PurchaseContractD

In [10]:
sold.columns

Index(['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BuildingAreaTotal', 'BedroomsTotal', 'C

### Task 2: Ensure numeric fields are properly typed

In [11]:
numeric_fields = [
    "OriginalListPrice",
    "ClosePrice",
    "Latitude",
    "Longitude",
    "LivingArea",
    "ListPrice",
    "DaysOnMarket",
    "ListingKeyNumeric",
    "ParkingTotal",
    "LotSizeAcres",
    "YearBuilt",
    "StreetNumberNumeric",
    "BathroomsTotalInteger",
    "BuyerAgencyCompensation",
    "BuildingAreaTotal",
    "BedroomsTotal",
    "BelowGradeFinishedArea",
    "Stories",
    "LotSizeArea",
    "MainLevelBedrooms",
    "GarageSpaces",
    "AssociationFee",
    "LotSizeSquareFeet",
    "rate_30yr_fixed"
]

In [12]:
for column in numeric_fields:
    if column in listing.columns:
        listing[column] = pd.to_numeric(listing[column], errors="coerce")

for column in numeric_fields:
    if column in sold.columns:
        sold[column] = pd.to_numeric(sold[column],errors="coerce")

### Task 3: Flag invalid numeric values

- ClosePRice <= 0
- LivingArea <= 0
- DaysOnMarket < 0
- Bathrooms or Bedrooms negative #

In [13]:
listing["invalid_close_price"] = (listing["ClosePrice"] <= 0)
listing["invalid_living_area"] = (listing["LivingArea"] <= 0)
listing["invalid_days_on_market"] = (listing["DaysOnMarket"] < 0)
listing["invalid_bedrooms"] = (listing["BedroomsTotal"] < 0)
listing["invalid_bathrooms"] = (listing["BathroomsTotalInteger"] < 0)

sold["invalid_close_price"] = (sold["ClosePrice"] <= 0)
sold["invalid_living_area"] = (sold["LivingArea"] <= 0)
sold["invalid_days_on_market"] = (sold["DaysOnMarket"] < 0)
sold["invalid_bedrooms"] = (sold["BedroomsTotal"] < 0)
sold["invalid_bathrooms"] = (sold["BathroomsTotalInteger"] < 0)

In [14]:
# summarize the flagged values

invalid_numeric_summary_listing = pd.Series({
    "ClosePrice <= 0":
        listing["invalid_close_price"].sum(),
    "LivingArea <= 0":
        listing["invalid_living_area"].sum(),
    "DaysOnMarket < 0":
        listing["invalid_days_on_market"].sum(),
    "BedroomsTotal < 0":
        listing["invalid_bedrooms"].sum(),
    "BathroomsTotalInteger < 0":
        listing["invalid_bathrooms"].sum()
})

invalid_numeric_summary_listing

ClosePrice <= 0                0
LivingArea <= 0              360
DaysOnMarket < 0              27
BedroomsTotal < 0              0
BathroomsTotalInteger < 0      0
dtype: int64

In [15]:
invalid_numeric_summary_sold = pd.Series({
    "ClosePrice <= 0":
        sold["invalid_close_price"].sum(),
    "LivingArea <= 0":
        sold["invalid_living_area"].sum(),
    "DaysOnMarket < 0":
        sold["invalid_days_on_market"].sum(),
    "BedroomsTotal < 0":
        sold["invalid_bedrooms"].sum(),
    "BathroomsTotalInteger < 0":
        sold["invalid_bathrooms"].sum()
})

invalid_numeric_summary_sold

ClosePrice <= 0                1
LivingArea <= 0              150
DaysOnMarket < 0              48
BedroomsTotal < 0              0
BathroomsTotalInteger < 0      0
dtype: int64

### Task 4: Date consistency Checks
- *listing_after_close_flag*
- *purchase_after_close_flag*
- *negative_timeline_flag*

Check the different dates to see if they align/are consistent

In [16]:
listing["listing_after_close_flag"] = (listing["ListingContractDate"] > listing["CloseDate"])
listing["purchase_after_close_flag"] = (listing["PurchaseContractDate"] >listing["CloseDate"])
listing["negative_timeline_flag"] = (listing["ListingContractDate"] > listing["CloseDate"]) + (listing["PurchaseContractDate"] >listing["CloseDate"])

listing.loc[:, ["listing_after_close_flag", "purchase_after_close_flag", "negative_timeline_flag"]]

,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
547157,False,False,False
547158,False,False,False
547159,False,False,False
547160,False,False,False


In [17]:
sold["listing_after_close_flag"] = (sold["ListingContractDate"] >= sold["CloseDate"])
sold["purchase_after_close_flag"] = (sold["PurchaseContractDate"] >= sold["CloseDate"])
sold["negative_timeline_flag"] = (sold["ListingContractDate"] >= sold["CloseDate"]) + (sold["PurchaseContractDate"] >= sold["CloseDate"])

sold.loc[:, ["listing_after_close_flag", "purchase_after_close_flag", "negative_timeline_flag"]]

,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
414049,False,False,False
414050,False,False,False
414051,False,False,False
414052,False,False,False


In [18]:
# check the number of rows in sold in the "listing_after_close_flag", "purchase_after_close_flag", "negative_timeline_flag" columns
sold[["listing_after_close_flag", "purchase_after_close_flag", "negative_timeline_flag"]].sum()

listing_after_close_flag     3891
purchase_after_close_flag    7581
negative_timeline_flag       7687
dtype: int64

In [19]:
sold[["listing_after_close_flag", "purchase_after_close_flag", "negative_timeline_flag"]].describe()

,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
count,414054,414054,414054
unique,2,2,2
top,False,False,False
freq,410163,406473,406367


### Task 5: Geographic Data Checks

Check the **Latitude** and the **Longitude** columns and flag values that meet this critera:
- Missing coordinates(lat or long is null)
- Lat = 0 or Long = 0 (sentinel null values)
- Long > 0 errors (California coordinates should be negative)
- Out-of-state or implausible coordinates

In [20]:
listing["missing_coordinates_flag"] = listing["Latitude"].isnull() | listing["Longitude"].isnull()
listing["zero_coordinates_flag"] = (listing["Latitude"] == 0) & (listing["Longitude"] == 0)
listing["positive_longitude_flag"] = listing["Longitude"] > 0

# for implausible coordinates, this is what i used: https://en.wikipedia.org/wiki/Module:Location_map/data/USA_California 
listing["implausible_coordinates_flag"] = (listing["Latitude"] < 32.3) | (listing["Latitude"] > 42.2) | (listing["Longitude"] > -113.9) | (listing["Longitude"] < -124.7)

In [21]:
sold["missing_coordinates_flag"] = sold["Latitude"].isnull() | sold["Longitude"].isnull()
sold["zero_coordinates_flag"] = (sold["Latitude"] == 0) & (sold["Longitude"] == 0)
sold["positive_longitude_flag"] = sold["Longitude"] > 0
sold["implausible_coordinates_flag"] = (sold["Latitude"] < 32.3) | (sold["Latitude"] > 42.2) | (sold["Longitude"] > -113.9) | (sold["Longitude"] < -124.7)

In [22]:
geo_summary_listing = pd.Series({
    "Missing coordinates":
        listing["missing_coordinates_flag"].sum(),

    "Zero coordinates":
        listing["zero_coordinates_flag"].sum(),

    "Positive longitude":
        listing["positive_longitude_flag"].sum(),

    "Implausible California coordinates":
        listing["implausible_coordinates_flag"].sum()
})

geo_summary_listing

Missing coordinates                   80247
Zero coordinates                         64
Positive longitude                       73
Implausible California coordinates      295
dtype: int64

In [23]:
geo_summary_sold = pd.Series({
    "Missing coordinates":
        sold["missing_coordinates_flag"].sum(),

    "Zero coordinates":
        sold["zero_coordinates_flag"].sum(),

    "Positive longitude":
        sold["positive_longitude_flag"].sum(),

    "Implausible California coordinates":
        sold["implausible_coordinates_flag"].sum()
})

geo_summary_sold

Missing coordinates                   4104
Zero coordinates                        27
Positive longitude                      31
Implausible California coordinates      91
dtype: int64

### Remove the flagged values

In [24]:
invalid_columns = [
    "invalid_close_price",
    "invalid_living_area",
    "invalid_days_on_market",
    "invalid_bedrooms",
    "invalid_bathrooms"
]

sold["invalid_numeric_flag"] = sold[invalid_columns].any(axis=1)
listing["invalid_numeric_flag"] = listing[invalid_columns].any(axis=1)

print("Sold Rows Flagged to Drop:", sold["invalid_numeric_flag"].sum())
print("Listing  Rows Flagged to Drop:", listing["invalid_numeric_flag"].sum())

Sold Rows Flagged to Drop: 199
Listing  Rows Flagged to Drop: 387


In [32]:
sold["ClosePrice"].isna().sum()

np.int64(2)

In [39]:
sold[sold["ClosePrice"].isna()][
    ["ListingKey", "ClosePrice", "ListPrice", "LivingArea"]
]

,ListingKey,ClosePrice,ListPrice,LivingArea
92206,1045745288,NaN,575000.0,2200.0
182382,1086646611,NaN,2700000.0,3109.0


In [40]:
sold_cleaned = sold[~sold["invalid_numeric_flag"]].copy()
listing_cleaned = listing[~listing["invalid_numeric_flag"]].copy()

print("Sold rows before cleaning:", len(sold))
print("Sold rows after cleaning:", len(sold_cleaned))

print("Listing rows before cleaning:", len(listing))
print("Listing rows after cleaning:", len(listing_cleaned))

Sold rows before cleaning: 414054
Sold rows after cleaning: 413855
Listing rows before cleaning: 547162
Listing rows after cleaning: 546775


In [27]:
listing_cleaned["invalid_numeric_flag"].value_counts()

invalid_numeric_flag
False    546775
Name: count, dtype: int64

In [28]:
sold_cleaned["invalid_numeric_flag"].value_counts()

invalid_numeric_flag
False    413855
Name: count, dtype: int64

In [29]:
report_dir = Path("../data/reports")
report_dir.mkdir(exist_ok=True)

invalid_numeric_summary_listing.to_csv(
    report_dir / "week4_5_invalid_numeric_summary.csv"
)

invalid_numeric_summary_sold.to_csv(
    report_dir / "week4_5_invalid_numeric_summary.csv"
)

geo_summary_listing.to_csv(
    report_dir / "week4_5_geographic_summary.csv"
)

geo_summary_sold.to_csv(
    report_dir / "week4_5_geographic_summary.csv"
)

In [41]:
output_dir = Path("../data/cleaned_csv")
output_dir.mkdir(exist_ok=True)

sold_cleaned.to_csv(
    output_dir / "sold_cleaned.csv",
    index=False
)

listing_cleaned.to_csv(
    output_dir / "listing_cleaned.csv",
    index=False
)